In [72]:
from typing import Literal
from pydantic import BaseModel, Field


class DishClassification(BaseModel):
    recipe_uid: str = Field(
        description="입력으로 받은 레시피의 recipe_uid를 그대로 반환"
    )

    dish_group: Literal[
        "국",
        "찌개",
        "탕",
        "전골",
        "조림",
        "볶음",
        "구이",
        "찜",
        "튀김",
        "전/부침",
        "무침",
        "샐러드",
        "밥",
        "죽",
        "면/국수",
        "절임/장아찌",
        "소스/양념",
        "빵/베이킹",
        "디저트",
        "음료",
        "기타",
    ] = Field(
        description="완성 음식의 대표적인 조리 형태"
    )

    dish_type: str = Field(
        description=(
            "레시피가 실제로 의미하는 대표 음식명. "
            "광고 문구, 조리 팁, 사람 이름, 황금레시피 등의 표현은 제거한다."
        )
    )

In [73]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

모델: gpt-5.6-luna


In [74]:
from pathlib import Path
import json

path = Path("input") / "recipes_10000_cleaned.jsonl"

recipes = []

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break

        recipes.append(json.loads(line))

print("파일 존재:", path.exists())
print("불러온 개수:", len(recipes))

파일 존재: True
불러온 개수: 5000


In [75]:
llm = make_model()

In [76]:
structured_llm = llm.with_structured_output(DishClassification)

In [77]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
너는 한국 음식 레시피를 지식 그래프용으로 분류하는 전문가다.

주어진 레시피를 분석해서 다음 두 가지를 결정한다.

1. dish_group
- 음식의 대표적인 조리 형태를 의미한다.
- 반드시 허용된 DishGroup 중 하나를 선택한다.
- 레시피 과정 중 잠깐 등장하는 조리법이 아니라,
  최종적으로 완성되는 음식의 형태를 기준으로 판단한다.

예시:
- 김치찌개 → 찌개
- 된장찌개 → 찌개
- 닭볶음탕 → 탕
- 갈비탕 → 탕
- 미역국 → 국
- 감자조림 → 조림
- 오징어볶음 → 볶음
- 제육볶음 → 볶음
- 갈비찜 → 찜
- 생선구이 → 구이


2. dish_type
- 실제 음식 종류를 나타내는 대표 음식명이다.
- 원본 title을 그대로 복사하지 않는다.
- 음식 이름만 남긴다.
- 같은 음식은 최대한 동일한 이름으로 통일한다.

dish_type에서 제거해야 하는 표현:
- 황금레시피
- 만드는 법
- 만들기
- 맛있게 만드는 법
- 초간단
- 백종원
- 엄마표
- 집밥
- 레시피
- 비법
- 추천
- 최고의
- 진짜진짜
- 쉽고 간단한
등 음식 자체의 이름이 아닌 표현

예시:
"돼지고기 김치찌개 맛내는 비법"
→ 김치찌개

"오징어 볶음, 향과 맛이 일품! 백종원 오징어 볶음"
→ 오징어볶음

"엄마의 레시피, 소고기 미역국 끓이는 법"
→ 소고기미역국

"닭볶음탕 진짜진짜 황금레시피 알려 드려요"
→ 닭볶음탕


중요한 규칙:

- title 자체는 수정하거나 생성하지 않는다.
- 입력에 없는 새로운 음식을 임의로 만들어내지 않는다.
- 음식 이름이 title에 명확하게 존재하면 title을 우선한다.
- title만으로 애매하면 description, 재료, 조리 과정을 참고한다.
- 주요 재료가 음식 정체성에 중요한 경우 dish_type에 포함한다.
  예: 소고기미역국, 참치김치찌개, 오징어볶음
- 단순한 홍보 문구나 조리 특징은 dish_type에서 제거한다.
- 준비 과정에서 잠깐 볶는다고 해서 무조건 '볶음'으로 분류하지 않는다.
- 최종 완성 음식의 형태를 기준으로 dish_group을 정한다.
- 정말 분류하기 어려운 경우에만 dish_group을 '기타'로 한다.

recipe_uid는 입력값을 한 글자도 변경하지 않고 그대로 반환한다.
"""
    ),
    (
        "human",
        """
[recipe_uid]
{recipe_uid}

[title]
{title}

[description]
{description}

[주요 재료]
{ingredients}

[조리 과정]
{steps}
"""
    )
])

In [78]:
classification_chain = prompt | structured_llm

In [79]:
def make_llm_input(recipe):
    # 재료명만 사용
    ingredient_names = [
        item["name_normalized"]
        for item in recipe.get("ingredients_clean", [])
        if item.get("name_normalized")
    ]

    # 양념도 음식 판단에 도움이 될 수 있으므로 이름만 추가
    seasoning_names = [
        item["name_normalized"]
        for item in recipe.get("seasonings_clean", [])
        if item.get("name_normalized")
    ]

    ingredient_names = list(dict.fromkeys(
        ingredient_names + seasoning_names
    ))

    # 조리 단계
    steps = recipe.get("steps", [])

    # 너무 긴 레시피는 앞 4단계 + 마지막 2단계만 사용
    if len(steps) > 6:
        selected_steps = steps[:4] + steps[-2:]
    else:
        selected_steps = steps

    steps_text = "\n".join(
        f"{step['order']}. {step['text']}"
        for step in selected_steps
    )

    # 지나치게 긴 텍스트 제한
    steps_text = steps_text[:2500]

    return {
        "recipe_uid": recipe["recipe_uid"],
        "title": recipe.get("title", ""),
        "description": recipe.get("description", ""),
        "ingredients": ", ".join(ingredient_names),
        "steps": steps_text,
    }

In [80]:
batch_inputs = [
    make_llm_input(recipe)
    for recipe in recipes
]

print(len(batch_inputs))

5000


In [ ]:
from tqdm.auto import tqdm


BATCH_SIZE = 100
MAX_CONCURRENCY = 10

classification_results = []
failed_results = []


for start in tqdm(range(0, len(batch_inputs), BATCH_SIZE)):

    batch = batch_inputs[start:start + BATCH_SIZE]

    results = classification_chain.batch(
        batch,
        config={
            "max_concurrency": MAX_CONCURRENCY
        },
        return_exceptions=True
    )

    for input_data, result in zip(batch, results):

        if isinstance(result, Exception):

            failed_results.append({
                "recipe_uid": input_data["recipe_uid"],
                "error": str(result)
            })

        else:
            classification_results.append(result)

  2%|▏         | 1/50 [00:05<04:28,  5.47s/it]

In [ ]:
for result in classification_results[:20]:
    print(
        result.recipe_uid,
        "/",
        result.dish_group,
        "/",
        result.dish_type
    )

10000recipe_6876357 / 탕 / 닭볶음탕
10000recipe_1785098 / 찌개 / 돼지고기김치찌개
10000recipe_6873683 / 국 / 소고기미역국
10000recipe_6903507 / 볶음 / 오징어볶음
10000recipe_6879215 / 볶음 / 소불고기
10000recipe_6879533 / 면/국수 / 잡채
10000recipe_6867256 / 무침 / 콩나물무침
10000recipe_6883937 / 탕 / 닭볶음탕
10000recipe_6912220 / 찌개 / 순두부찌개
10000recipe_6905743 / 볶음 / 제육볶음
10000recipe_6845428 / 볶음 / 제육볶음
10000recipe_6905196 / 국 / 콩나물국
10000recipe_6894096 / 기타 / 떡볶이
10000recipe_6897261 / 무침 / 오이무침
10000recipe_6880798 / 찜 / 안동찜닭
10000recipe_6906655 / 조림 / 두부조림
10000recipe_6891526 / 조림 / 두부조림
10000recipe_6891816 / 볶음 / 멸치볶음
10000recipe_6859263 / 찌개 / 된장찌개
10000recipe_6893285 / 무침 / 무생채


In [ ]:
classification_map = {
    result.recipe_uid: result
    for result in classification_results
}

In [ ]:
print(len(recipes))
print(recipes[0].keys())
print(recipes[0].get("recipe_uid"))

4016
dict_keys(['schema_version', 'recipe', 'dish', 'components'])
None


In [ ]:
import json
from pathlib import Path

path = Path("input") / "recipes_10000_cleaned.jsonl"

recipes = []

with path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break

        recipes.append(json.loads(line))

print("개수:", len(recipes))
print("첫 uid:", recipes[0]["recipe_uid"])

개수: 5000
첫 uid: 10000recipe_6876357


In [ ]:
classified_recipes = []

for recipe in recipes:

    uid = recipe["recipe_uid"]

    result = classification_map.get(uid)

    new_recipe = recipe.copy()

    if result is not None:
        new_recipe["dish_group"] = result.dish_group
        new_recipe["dish_type"] = result.dish_type
    else:
        new_recipe["dish_group"] = None
        new_recipe["dish_type"] = None

    classified_recipes.append(new_recipe)

In [ ]:
print(
    classified_recipes[0]["title"],
    classified_recipes[0]["dish_group"],
    classified_recipes[0]["dish_type"]
)


닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^ 탕 닭볶음탕


In [ ]:

output_path = Path("input/recipes_5000_classified.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for recipe in classified_recipes:
        f.write(
            json.dumps(
                recipe,
                ensure_ascii=False
            ) + "\n"
        )

print(output_path)

input\recipes_5000_classified.jsonl


In [ ]:
output_path = Path("input/recipes_5000_classified.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for recipe in classified_recipes:
        f.write(
            json.dumps(
                recipe,
                ensure_ascii=False
            ) + "\n"
        )

print(output_path)

NameError: name 'Path' is not defined

In [ ]:
print(classified_recipes[0]["title"])

닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^


In [ ]:
for recipe in classified_recipes:
    steps = recipe.get("steps", [])

    # order 순서대로 정렬
    steps = sorted(
        steps,
        key=lambda x: x.get("order", 9999)
    )

    # 조리법을 하나의 문자열로 합치기
    cooking_method = "\n".join(
        f"{step['order']}. {step['text'].strip()}"
        for step in steps
        if step.get("text")
    )

    recipe["cooking_method"] = cooking_method

In [ ]:
import json
from pathlib import Path

output_path = Path("input") / "recipes_5000_classified.jsonl"

with output_path.open("w", encoding="utf-8") as f:
    for recipe in classified_recipes:
        f.write(
            json.dumps(recipe, ensure_ascii=False) + "\n"
        )

print("저장 완료:", output_path)

저장 완료: input\recipes_5000_classified.jsonl


In [ ]:
import json
from pathlib import Path

path = Path("input") / "recipes_5000_classified.jsonl"

classified_recipes = []

with path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            classified_recipes.append(json.loads(line))

print("불러온 개수:", len(classified_recipes))
print(classified_recipes[0]["title"])
print(classified_recipes[0]["dish_group"])
print(classified_recipes[0]["dish_type"])

불러온 개수: 5000
닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^
탕
닭볶음탕


In [ ]:
null_group = [
    recipe
    for recipe in classified_recipes
    if recipe.get("dish_type") is None
]

print("dish_group null 개수:", len(null_group))

dish_group null 개수: 0


In [ ]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Neo4j 연결 성공")

Neo4j 연결 성공


In [ ]:
invalid_recipes = [
    recipe
    for recipe in classified_recipes
    if not recipe.get("dish_group")
    or not recipe.get("dish_type")
]

print("전체:", len(classified_recipes))
print("분류 정상:", len(classified_recipes) - len(invalid_recipes))
print("분류 누락:", len(invalid_recipes))

전체: 5000
분류 정상: 4996
분류 누락: 4


In [ ]:
for recipe in invalid_recipes[:10]:
    print(
        recipe.get("recipe_uid"),
        "|",
        recipe.get("title"),
        "|",
        recipe.get("dish_group"),
        "|",
        recipe.get("dish_type")
    )

10000recipe_6845117 | None | 기타 | 
10000recipe_6862096 | None | 기타 | 
10000recipe_6846102 | None | 기타 | 
10000recipe_6906984 | None | 기타 | 


In [ ]:
def format_number(value):
    """200.0 -> '200', 0.5 -> '0.5'"""
    if value is None:
        return None

    if float(value).is_integer():
        return str(int(value))

    return str(value)


def get_quantity(item):
    """
    cleaned 데이터에서 관계의 quantity 생성.

    200g     -> quantity='200'
    1~1.5개  -> quantity='1~1.5'
    약간     -> quantity='약간'
    """

    qualitative = item.get("qualitative_amount")

    if qualitative:
        return qualitative

    amount_min = item.get("amount_min")
    amount_max = item.get("amount_max")

    if amount_min is not None:

        if amount_max is None or amount_min == amount_max:
            return format_number(amount_min)

        return (
            f"{format_number(amount_min)}"
            f"~{format_number(amount_max)}"
        )

    return None

In [ ]:
def make_graph_row(recipe):

    recipe_uid = recipe["recipe_uid"]

    # -------------------------
    # Dish
    # -------------------------

    dish = {
        "recipe_uid": recipe_uid,
        "title": recipe.get("title"),
        "views": recipe.get("views"),
        "servings": recipe.get("servings"),
        "cooking_time": recipe.get("cooking_time"),
        "difficulty": recipe.get("difficulty"),
        "source_url": recipe.get("source_url"),
        "cooking_method": recipe.get("cooking_method"),
    }

    ingredients = []

    # -------------------------
    # 일반 재료
    # -------------------------

    for idx, item in enumerate(
        recipe.get("ingredients_clean", [])
    ):

        name = item.get("name_normalized")

        if not name:
            continue

        ingredients.append({
            "usage_key": (
                f"{recipe_uid}|food|{idx}|{name}"
            ),

            "name_normalized": name,

            "quantity": get_quantity(item),

            "unit": item.get("unit_normalized"),

            "preparation": item.get("preparation") or [],

            "role": "food",

            "raw": item.get("raw"),

            "amount_text": item.get("amount_text"),

            "source_group": item.get("group"),
        })

    # -------------------------
    # 양념
    # -------------------------

    offset = len(ingredients)

    for idx, item in enumerate(
        recipe.get("seasonings_clean", [])
    ):

        name = item.get("name_normalized")

        if not name:
            continue

        ingredients.append({
            "usage_key": (
                f"{recipe_uid}|seasoning|{offset + idx}|{name}"
            ),

            "name_normalized": name,

            "quantity": get_quantity(item),

            "unit": item.get("unit_normalized"),

            "preparation": item.get("preparation") or [],

            "role": "seasoning",

            "raw": item.get("raw"),

            "amount_text": item.get("amount_text"),

            "source_group": item.get("group"),
        })

    dish_group = recipe["dish_group"]
    dish_type = recipe["dish_type"]

    return {
        "dish_group": dish_group,

        "dish_type": dish_type,

        # 같은 이름의 DishType이 다른 대분류에 생기는 문제 방지
        "dish_type_key": f"{dish_group}::{dish_type}",

        "dish": dish,

        "ingredients": ingredients,
    }

In [ ]:
graph_rows = []

for recipe in classified_recipes:

    if (
        not recipe.get("dish_group")
        or not recipe.get("dish_type")
    ):
        continue

    graph_rows.append(
        make_graph_row(recipe)
    )


print("Neo4j 적재 대상:", len(graph_rows))

Neo4j 적재 대상: 4996


In [ ]:
graph_rows[0]

{'dish_group': '탕',
 'dish_type': '닭볶음탕',
 'dish_type_key': '탕::닭볶음탕',
 'dish': {'recipe_uid': '10000recipe_6876357',
  'title': '닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^',
  'views': 6199000,
  'servings': '3인분',
  'cooking_time': '60분 이내',
  'difficulty': '초급',
  'source_url': 'https://www.10000recipe.com/recipe/6876357',
  'cooking_method': '1. 닭 한마리를 준비 합니다 저희집은 가족이 5명이라 두마리를 준비했어요 대식가거든요 ㅎㅎ\n2. 당면을 물에 불려 줍니다\n3. 닭을 깨끗이 씻어 손질 해 줍니다\n4. 닭 사이사이 붙어있는 기름을 제거해주면 좋아요\n5. 잡내를 없애기 위해 우유에 30분정도 담가 둡니다\n6. 지금부터 이부분은 해도되고 건너 뛰어도 되는 과정 인데요 닭을 한번후루룩 끓여 줍니다\n7. 닭끓인 물을 버리고 차가운물로 헹궈줍니다 이부분에서 제가 두가지를 해봤는데요 1.닭을 우유에 담갔다가 씻은후 삶지않고 조리한경우 2.닭을우유에 담갔다가 삶아서 조리한 경우 한번 삶아서 육수를 버리고난 닭볶음탕은 깊은맛이 없더라구요 닭의 잡내를 잡아주는건 좋은데 그냥유유에만 한번 담가주는 방법이 훨씬 진하고 맛있다는 사실 ㅎ 무튼 취향대로 하시면 좋을것 같아요~~^^\n8. 고기를 냄비에두고 닭한마리기준 간장9~11 고추장4 고춧가루2 양파갈은거1/2 마늘5알 요리당6 청주1 후추약간 액젓1 액젓을 넣으면 감칠맛이 살아나요. 단, 싱겁게 드시는 분들은 간장을 줄이시고 액젓을 추가해요.\n9. 이렇게 양념소스를 만들어 주세요 요거 진짜 맛집 하고 거의 흡사한 맛이에요 맛집은 낙지나 새우를 첨가해 주기도 하죠!\n10. 냄비에 양념장과 육수600ml를 넣고 한소끔 끓여 줍니다 물대신 육수를 넣어주시면 좀더 감칠

In [ ]:
constraints = [
    """
    CREATE CONSTRAINT dish_uid_unique IF NOT EXISTS
    FOR (d:Dish)
    REQUIRE d.recipe_uid IS UNIQUE
    """,

    """
    CREATE CONSTRAINT dish_group_unique IF NOT EXISTS
    FOR (g:DishGroup)
    REQUIRE g.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT dish_type_unique IF NOT EXISTS
    FOR (t:DishType)
    REQUIRE t.key IS UNIQUE
    """,

    """
    CREATE CONSTRAINT ingredient_unique IF NOT EXISTS
    FOR (i:Ingredient)
    REQUIRE i.name_normalized IS UNIQUE
    """
]


with driver.session(database=NEO4J_DATABASE) as session:

    for query in constraints:
        session.run(query).consume()


print("Constraint 생성 완료")

Constraint 생성 완료


In [ ]:
LOAD_QUERY = """
UNWIND $rows AS row

// -----------------------------
// DishGroup
// -----------------------------

MERGE (g:DishGroup {
    name: row.dish_group
})


// -----------------------------
// DishType
// -----------------------------

MERGE (t:DishType {
    key: row.dish_type_key
})

SET t.name = row.dish_type

MERGE (g)-[:HAS_TYPE]->(t)


// -----------------------------
// Dish
// -----------------------------

MERGE (d:Dish {
    recipe_uid: row.dish.recipe_uid
})

SET d.title = row.dish.title,
    d.views = row.dish.views,
    d.servings = row.dish.servings,
    d.cooking_time = row.dish.cooking_time,
    d.difficulty = row.dish.difficulty,
    d.source_url = row.dish.source_url,
    d.cooking_method = row.dish.cooking_method

MERGE (t)-[:HAS_DISH]->(d)


// -----------------------------
// Ingredient
// -----------------------------

WITH row, d

UNWIND row.ingredients AS ing

MERGE (i:Ingredient {
    name_normalized: ing.name_normalized
})

MERGE (d)-[r:INGREDIENT {
    usage_key: ing.usage_key
}]->(i)

SET r.quantity = ing.quantity,
    r.unit = ing.unit,
    r.preparation = ing.preparation,
    r.role = ing.role,
    r.raw = ing.raw,
    r.amount_text = ing.amount_text,
    r.source_group = ing.source_group
"""

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 200


with driver.session(database=NEO4J_DATABASE) as session:

    for start in tqdm(
        range(0, len(graph_rows), BATCH_SIZE)
    ):

        batch = graph_rows[
            start:start + BATCH_SIZE
        ]

        session.run(
            LOAD_QUERY,
            rows=batch
        ).consume()


print("Neo4j 적재 완료")

c:\Users\Playdata\Desktop\mle-01-p2-team2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 25/25 [00:08<00:00,  2.83it/s]

Neo4j 적재 완료
